1.  Explain how implicit missing values impact medical datasets, and justify your 
choice of imputation strategy when handling these zeros compared to outright 
row deletion.  

Implicit missing values are important in medical data because zeros in Glucose, BloodPressure, SkinThickness, Insulin, and BMI represent missing measurements rather than valid readings. Since isnull() doesn't detect these zeros, they can distort statistics and negatively affect models like Logistic Regression and SVC. I replaced these zeros with NaN and used median imputation because it handles skewed data and outliers better than the mean, especially in columns like Insulin. I also included the imputer in the pipeline so it was fitted only on the training data, preventing data leakage. Dropping rows would have removed valuable information from other features. Although median imputation reduces variance and doesn't account for relationships between features, it preserves more useful data than deleting incomplete rows.


2.  Contrast how hard voting aggregates predictions versus soft voting. Why do 
well-calibrated probability estimates matter when implementing soft voting 
across heterogeneous models? 

Hard voting combines predictions by having each model vote for a class, with the majority vote determining the final prediction. Soft voting instead averages the predicted probabilities from each model and selects the class with the highest average probability. Well calibrated probabilities are important because different models, such as Logistic Regression, SVC, and Decision Trees, may assign different confidence levels to their predictions. If a model is overconfident, it can have too much influence on the final result, even when its predictions are incorrect. Proper calibration makes the probabilities more reliable and allows soft voting to combine predictions more fairly and accurately.


3.  What is the role of out-of-bag evaluation in Bagging classifiers, and why does 
an OOB score serve as an efficient alternative to traditional cross-validation 
splits during ensemble training? 

In a Bagging classifier, each tree is trained on a random bootstrap sample drawn with replacement, meaning about 63% of the training rows are unique while the remaining 37% are left out. These left out rows are called out-of-bag (OOB) samples and can be used to evaluate the model without a separate validation set. Each row is predicted using only the trees that did not train on it, giving an estimate of how well the model generalizes to unseen data. OOB evaluation is more efficient than k-fold cross-validation because it requires only one training run instead of repeatedly fitting the entire ensemble. It also allows all 768 rows to remain available for training. In my results, the OOB score was 0.761 compared to a test accuracy of 0.734, a difference of just 0.027, showing that OOB evaluation provided a reasonable estimate of the model's performance.

4.  How does introducing random feature subsetting in Random Forests differ 
from standard Bagging, and how does this feature diversity impact model 
variance and generalization? 

Standard Bagging trains each tree on a random bootstrap sample while allowing every feature to be considered at each split. Random Forests add another layer of randomness by limiting each split to a random subset of features. In my model, max_features="sqrt" meant about 3 of the 8 features were considered at each split. This prevents dominant features like Glucose from being selected by nearly every tree, making the trees less similar and reducing the ensemble's variance. Although individual trees may become slightly weaker, the overall model often generalizes better. In my results, Random Forest achieved 0.7792 test accuracy compared to 0.734 for Bagging, with Glucose remaining the most important predictor.

5.  Discuss the mechanics of sequential sample weighting in AdaBoost. Based 
on your training versus test accuracy results, did AdaBoost exhibit signs of 
overfitting, and how can hyperparameter adjustments mitigate this?

AdaBoost builds its ensemble sequentially by initially assigning equal weights to all training samples and fitting a decision stump. After each round, it calculates the stump's weighted error and increases the weights of misclassified samples so the next stump focuses more on difficult cases. Each stump also receives a voting weight based on its accuracy, with better-performing stumps having more influence on the final prediction. In my results, AdaBoost achieved about 0.81 training accuracy and 0.73 test accuracy after 200 rounds. Test accuracy peaked around 0.76 near round 80 before gradually declining, suggesting mild overfitting as later rounds began focusing on noisy or difficult samples. Lowering learning_rate, reducing n_estimators, or keeping the base learners shallow could help limit overfitting. These adjustments should be evaluated using cross-validation rather than the test set to avoid overly optimistic results.